<a href="https://colab.research.google.com/github/ClassNeuralNetwork/Chatbot-casos-de-teste/blob/main/ChatBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re
import time

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
def detectar_entrada(texto):
  """
  Detectar automaticamente se o input é código ou requisito textual.
  Retornar 'codigo' ou 'requisito'
  """

  padroes_codigo = [
        r"\bdef\s+\w+\s*\(",
        r"\bfunction\s+\w+\s*\(",
        r"\bclass\s+\w+",
        r"\bpublic\s+(static\s+)?\w+",
        r"\bvoid\s+\w+\s*\(",
        r"\breturn\s+",
        r"\bif\s*\(.+\)\s*[:{]",
        r"=>\s*[{(]",
        r"\bimport\s+\w+",
        r"#include\s*<",
        r"\bprint\s*\(",
        r"::\w+",
  ]

  for padrao in padroes_codigo:
    if re.search(padrao, texto):
      return "codigo"

  return "requisito"

In [ ]:
def gerar_casos_teste(requisito):
    """Gera casos de teste em tabela a partir de um requisito textual."""
    messages = [
        {
            "role": "system",
            "content": """
            Generate ONLY software test cases based on the given requirement.
            Output format:

            | ID | Test Type | Scenario | Input | Expected Result |

            Include:
            - Positive tests
            - Negative tests
            - Boundary tests
            """
        },
        {
            "role": "user",
            "content": requisito
        }
    ]

    return chamar_modelo(messages)


def gerar_codigo_teste(codigo):
  """Gera código de testes automatizados (pytest/unittest) a partir de um CÓDIGO fornecido."""

  messages = [
        {
            "role": "system",
            "content": """
            You are a test engineer. Given a code snippet, generate automated test code using pytest.

            Rules:
            - Write complete, runnable pytest test functions
            - Cover: happy path, edge cases, invalid inputs, boundary values
            - Use descriptive test function names (test_<scenario>)
            - Add brief comments explaining each test
            - Output ONLY the test code, no explanations outside the code
            """
        },
        {
            "role": "user",
            "content": f"Generate tests for the following code:\n\n{codigo}"
        }
]

  return chamar_modelo(messages)

historico_metricas = []

def chamar_modelo(messages):
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    tokens_entrada = inputs.input_ids.shape[1]

    t0 = time.perf_counter()

    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=500,
        temperature=0.2
    )

    resposta = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    t1 = time.perf_counter()

    tokens_saida = outputs.shape[1] - tokens_entrada
    tempo_total = round(t1 - t0, 2)
    tok_por_seg = round(tokens_saida / tempo_total, 1) if tempo_total > 0 else 0.0

    historico_metricas.append({
        "tokens_entrada": tokens_entrada,
        "tokens_saida":   tokens_saida,
        "tempo":          tempo_total,
        "tok_seg":        tok_por_seg,
    })

    print(f"  ⏱  {tempo_total}s  |  "
          f"entrada: {tokens_entrada} tok  |  "
          f"saída: {tokens_saida} tok  |  "
          f"{tok_por_seg} tok/s")

    resposta = tokenizer.decode(
        outputs[0][tokens_entrada:],
        skip_special_tokens=True
    )

    return resposta

In [ ]:
def exibir_resumo():
    if not historico_metricas:
        print("Nenhuma chamada registrada ainda.")
        return

    print(f"\n\n{'='*65}")
    print("📊 RESUMO DE DESEMPENHO DO MODELO")
    print(f"{'='*65}")
    print(f"{'#':>3}  {'Entrada':>8}  {'Saída':>6}  {'Tempo':>7}  {'tok/s':>7}")
    print("-"*65)

    for i, m in enumerate(historico_metricas, 1):
        print(f"{i:>3}  {m['tokens_entrada']:>8}  {m['tokens_saida']:>6}  "
              f"{m['tempo']:>6}s  {m['tok_seg']:>7}")

    print("-"*65)

    n = len(historico_metricas)
    med_entrada = sum(m["tokens_entrada"] for m in historico_metricas) / n
    med_saida   = sum(m["tokens_saida"]   for m in historico_metricas) / n
    med_tempo   = sum(m["tempo"]          for m in historico_metricas) / n
    med_toks    = sum(m["tok_seg"]        for m in historico_metricas) / n

    print(f"{'Média':>3}  {med_entrada:>8.1f}  {med_saida:>6.1f}  "
          f"{med_tempo:>6.2f}s  {med_toks:>7.1f}")
    print(f"{'='*65}\n")



In [ ]:
print("=== Chatbot de Casos de Teste ===")
print("- Cole um REQUISITO  → gera tabela de casos de teste")
print("- Cole um CÓDIGO     → gera código de testes (pytest)")
print("- Digite 'sair'      → encerra")

while True:
    print("\n" + "-" * 50)
    entrada = input("Entrada: ")

    if entrada.strip().lower() == "sair":
        print("Encerrando...")
        break

    if entrada.strip().lower() == "resumo":
        exibir_resumo()
        continue

    if not entrada.strip():
        continue

    tipo = detectar_entrada(entrada)

    if tipo == "codigo":
        print("\n[Detectado: CÓDIGO → gerando testes automatizados com pytest]\n")
        resposta = gerar_codigo_teste(entrada)
    else:
        print("\n[Detectado: REQUISITO → gerando tabela de casos de teste]\n")
        resposta = gerar_casos_teste(entrada)

    print(resposta)

=== Chatbot de Casos de Teste ===
- Cole um REQUISITO  → gera tabela de casos de teste
- Cole um CÓDIGO     → gera código de testes (pytest)
- Digite 'sair'      → encerra

--------------------------------------------------

[Detectado: CÓDIGO → gerando testes automatizados com pytest]

  ⏱  20.86s  |  entrada: 197 tok  |  saída: 500 tok  |  24.0 tok/s
```python
import pytest

class TestCadastrarLivro:
    
    @pytest.fixture
    def setup(self):
        # Setup code here
        pass
    
    def test_cadastra_livro_with_valid_inputs(self, setup):
        # Happy Path Scenario
        titulo = "Test Title"
        autor = "Test Author"
        isbn = "1234567890"
        ano = 2023
        quantidade = 1
        
        resultado = cadastrar_livro(titulo, autor, isbn, ano, quantidade)
        
        assert isinstance(resultado, str), "Resultado deve ser uma string."
        assert resultado == "Livro cadastrado com sucesso!", "Resultado deve ser 'Livro cadastrado com sucesso!'"

 